# 02 - Feature engineering (next-week forecast)
**Forecast setting:** on the row for week *t* we know everything up to the end of week *t* (this week's cases and weather) and predict week *t+1*.
**Target:** `target_next_cases` = next week's cases, same district; we train on `target_log = log1p(...)` and convert back with `expm1`.
**Allowed:** this week and the past. **Never allowed:** next week or later.
**Dropped:** `is_epidemic_year`, `incidence_rate`, `cases_zscore` (leakage / copies of cases). The file's `cases_lag1`, `rainfall_lag2`, `rainfall_lag4`, `year` and `week` are rebuilt from the date.
The code lives in `src/features.py`; this notebook runs and checks it.

In [1]:
# AI-GENERATED (Claude via Antigravity)
import sys, numpy as np, pandas as pd
sys.path.append("../src")
from features import load_raw, build_features, DISTRICT, DATE, CASES
raw = load_raw("../data/raw/dengue_srilanka_full_2006_2025.csv")
feat, groups = build_features(raw)
features = [c for cols in groups.values() for c in cols]
print("raw rows:", len(raw), "-> feature rows:", len(feat), "| dropped:", len(raw) - len(feat))
print("missing values left in features/target:", int(feat[features + ["target_log"]].isna().sum().sum()))

raw rows: 24775 -> feature rows: 24550 | dropped: 225
missing values left in features/target: 0


## Checking the dropped columns
- Does `cases_zscore` use the **whole series** (including future weeks)? We recompute it from each district's mean and standard deviation over all years. An exact match means it leaks.
- Are my rebuilt rainfall lags the same as the file's `rainfall_lag2/4` and `cases_lag1`? If yes, dropping the originals loses nothing.

In [2]:
# AI-GENERATED (Claude via Antigravity)
g = raw.groupby(DISTRICT)[CASES]
z_all_years = (raw[CASES] - g.transform("mean")) / g.transform(lambda s: s.std(ddof=0))
print("cases_zscore vs district mean/std over ALL years, max difference:", float((z_all_years - raw["cases_zscore"]).abs().max()))
sh = lambda col, k: raw.groupby(DISTRICT)[col].shift(k)
print("rainfall_lag2 matches rebuilt lag 2:", bool(((sh("rainfall_mm", 2) - raw["rainfall_lag2"]).abs().fillna(0) < 1e-9).all()))
print("rainfall_lag4 matches rebuilt lag 4:", bool(((sh("rainfall_mm", 4) - raw["rainfall_lag4"]).abs().fillna(0) < 1e-9).all()))
print("cases_lag1 matches rebuilt lag 1:   ", bool((sh(CASES, 1).fillna(-1) == raw["cases_lag1"].fillna(-1)).all()))

cases_zscore vs district mean/std over ALL years, max difference: 0.0
rainfall_lag2 matches rebuilt lag 2: True
rainfall_lag4 matches rebuilt lag 4: True
cases_lag1 matches rebuilt lag 1:    True


## Sanity check: the series is evenly spaced, so 'one row back' really is 'one week back'

In [3]:
# AI-GENERATED (Claude via Antigravity)
step = raw.groupby(DISTRICT)[DATE].diff().dt.days.dropna()
print(step.value_counts().to_dict(), "(7 = normal; 6 and 8 are the date jitter seen in the audit)")
assert step.between(6, 8).all()

{7.0: 24700, 8.0: 25, 6.0: 25} (7 = normal; 6 and 8 are the date jitter seen in the audit)


## Tests (asserts)
For 3 random rows:
1. `cases_lag_0` is this week's cases, `cases_lag_1` is last week's cases **of the same district**, and the target is next week's cases.
2. **Leak test:** change the cases AND the weather of every week after *t* (+1000 cases, weather x2) and rebuild everything. Every feature on row *t* must stay identical. The target must change (proves the test can detect a change).
3. Changing week *t* itself must change `cases_lag_0` (it really uses this week).

In [4]:
# AI-GENERATED (Claude via Antigravity)
rng = np.random.default_rng(42)
picks = feat.iloc[rng.choice(len(feat), 3, replace=False)]
key = lambda f, d, t: f[(f[DISTRICT] == d) & (f[DATE] == t)].iloc[0]   # find one row by district + date
weather = ["rainfall_mm", "temp_max_c", "temp_min_c", "humidity_pct"]
for _, r in picks.iterrows():
    i = raw.index[(raw[DATE] == r[DATE]) & (raw[DISTRICT] == r[DISTRICT])][0]
    prev, nxt = raw.loc[i - 1], raw.loc[i + 1]
    assert prev[DISTRICT] == nxt[DISTRICT] == r[DISTRICT]                       # neighbours are the same district
    assert r["cases_lag_0"] == raw.loc[i, CASES], "lag_0 is not this week's cases"
    assert r["cases_lag_1"] == prev[CASES], "lag_1 is not last week's cases"
    assert r["target_next_cases"] == nxt[CASES], "target is not next week's cases"

    # Leak test: damage ALL later weeks of this district, rebuild, compare this row
    future = (raw[DISTRICT] == r[DISTRICT]) & (raw[DATE] > r[DATE])
    changed = raw.copy(); changed.loc[future, CASES] += 1000; changed.loc[future, weather] *= 2
    f2, _ = build_features(changed)
    old, new = key(feat, r[DISTRICT], r[DATE]), key(f2, r[DISTRICT], r[DATE])
    for c in features:
        assert old[c] == new[c], f"feature {c} uses future information!"
    assert new["target_next_cases"] == old["target_next_cases"] + 1000          # the target did change

    # Changing this week must change lag_0
    changed2 = raw.copy(); changed2.loc[i, CASES] += 1000
    assert key(build_features(changed2)[0], r[DISTRICT], r[DATE])["cases_lag_0"] == old["cases_lag_0"] + 1000
    print("OK:", r[DISTRICT], r[DATE].date())
print("All tests passed")

OK: Matara 2013-12-21


OK: Badulla 2011-06-18


OK: Nuwara Eliya 2013-08-31
All tests passed


## Save and list the features by type

In [5]:
# AI-GENERATED (Claude via Antigravity)
feat.to_csv("../data/processed/features.csv", index=False)
print("Saved data/processed/features.csv", feat.shape)
for name, cols in groups.items():
    print(f"\n{name} ({len(cols)}): {cols}")
print("\nNot model inputs (kept for splitting / scoring):", [DATE, CASES, "target_next_cases", "target_log"])
print("Total features:", len(features))

Saved data/processed/features.csv (24550, 55)

case lags (6): ['cases_lag_0', 'cases_lag_1', 'cases_lag_2', 'cases_lag_3', 'cases_lag_4', 'cases_lag_8']

rolling cases (4): ['cases_roll_mean_4', 'cases_roll_max_4', 'cases_roll_mean_8', 'cases_roll_max_8']

climate (36): ['rainfall_mm_lag_0', 'rainfall_mm_lag_1', 'rainfall_mm_lag_2', 'rainfall_mm_lag_3', 'rainfall_mm_lag_4', 'rainfall_mm_lag_5', 'rainfall_mm_lag_6', 'rainfall_mm_lag_7', 'rainfall_mm_lag_8', 'temp_max_c_lag_0', 'temp_max_c_lag_1', 'temp_max_c_lag_2', 'temp_max_c_lag_3', 'temp_max_c_lag_4', 'temp_max_c_lag_5', 'temp_max_c_lag_6', 'temp_max_c_lag_7', 'temp_max_c_lag_8', 'temp_min_c_lag_0', 'temp_min_c_lag_1', 'temp_min_c_lag_2', 'temp_min_c_lag_3', 'temp_min_c_lag_4', 'temp_min_c_lag_5', 'temp_min_c_lag_6', 'temp_min_c_lag_7', 'temp_min_c_lag_8', 'humidity_pct_lag_0', 'humidity_pct_lag_1', 'humidity_pct_lag_2', 'humidity_pct_lag_3', 'humidity_pct_lag_4', 'humidity_pct_lag_5', 'humidity_pct_lag_6', 'humidity_pct_lag_7', 'hu

## Reference: the persistence baseline (next week = this week) on all rows

In [6]:
# AI-GENERATED (Claude via Antigravity)
d = feat["target_next_cases"] - feat[CASES]
print("Persistence MAE (all rows, original scale): %.2f | RMSE: %.2f" % (d.abs().mean(), np.sqrt((d**2).mean())))

Persistence MAE (all rows, original scale): 10.92 | RMSE: 28.06
